In [2]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

In [3]:
# Load and sort data for a time-based split
df = pd.read_csv("startup_valuation_dataset.csv")
df["funding_date"] = pd.to_datetime(df["funding_date"], errors="coerce")
df = df.dropna(subset=["funding_date"]).sort_values("funding_date").reset_index(drop=True)

# Binary target
df["exited"] = df["exited"].astype(int)

# Early-stage feature derived from existing columns
df["startup_age_at_funding"] = df["funding_date"].dt.year - df["founded_year"]

# Drop identifiers and obvious leakage
drop_cols = [
    "startup_id",
    "startup_name",
    "exit_type",
    "funding_date",
    "estimated_revenue_usd",
    "estimated_valuation_usd"
]

X = df.drop(columns=drop_cols + ["exited"])
y = df["exited"]

# Time-based train/test split
split_idx = int(0.8 * len(df))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Separate numeric and categorical columns
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

# Preprocess numeric and categorical features
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Helper function for evaluation
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_score)
        pr_auc = average_precision_score(y_test, y_score)
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    print(name)
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
    print("F1:", round(f1_score(y_test, y_pred, zero_division=0), 4))
    print("ROC-AUC:", round(roc_auc, 4))
    print("PR-AUC:", round(pr_auc, 4))
    print()

# Baseline model: always predict the majority class
baseline_model = DummyClassifier(strategy="most_frequent")
evaluate_model("Baseline Model", baseline_model, X_train, y_train, X_test, y_test)

# Logistic regression pipeline with L2 regularization
logreg_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(
        penalty="l2",
        solver="liblinear",
        max_iter=2000,
        class_weight="balanced"
    ))
])

# Tune regularization strength with time-aware CV
param_grid = {"classifier__C": [0.01, 0.1, 1, 10, 100]}
tscv = TimeSeriesSplit(n_splits=5)

grid = GridSearchCV(
    estimator=logreg_pipeline,
    param_grid=param_grid,
    scoring="average_precision",
    cv=tscv,
    n_jobs=-1,
    refit=True
)

# Train and evaluate best logistic regression model
grid.fit(X_train, y_train)
best_logreg = grid.best_estimator_

print("Best C:", grid.best_params_["classifier__C"])
print()

evaluate_model("Logistic Regression with L2", best_logreg, X_train, y_train, X_test, y_test)

Baseline Model
Confusion Matrix:
 [[8502    0]
 [1498    0]]
Accuracy: 0.8502
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.5
PR-AUC: 0.1498

Best C: 1

Logistic Regression with L2
Confusion Matrix:
 [[5938 2564]
 [1026  472]]
Accuracy: 0.641
Precision: 0.1555
Recall: 0.3151
F1: 0.2082
ROC-AUC: 0.511
PR-AUC: 0.1536

